In [ ]:
!pip install -q peft transformers datasets accelerate ftfy regex Pillow tqdm

In [ ]:
import os
import math
import random
import json
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPTokenizer
import copy

from peft import LoraConfig, get_peft_model
from datasets import load_dataset, concatenate_datasets
from tqdm.auto import tqdm

# ── Detect dual T4 ──────────────────────────────────────────────────────────
N_GPU  = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPUs available: {N_GPU}  |  device: {DEVICE}")

In [ ]:
# ── Random permutation + word rotation ──────────────────────────────────────
ALPHABET = "abcdefghijklmnopqrstuvwxyz"

KEY = "cixmpuqbstreyofwlkzhgnavdj"   # <── Your generated key

def _validate_key(key: str) -> str:
    key = key.strip().lower()
    assert sorted(key) == list(ALPHABET), (
        "KEY must be a permutation of all 26 lowercase letters. "
    )
    return key

KEY = _validate_key(KEY)

_ENC_TABLE = str.maketrans(ALPHABET + ALPHABET.upper(), KEY + KEY.upper())
_DEC_TABLE = str.maketrans(KEY + KEY.upper(), ALPHABET + ALPHABET.upper())

def _rotate(word: str) -> str:
    return (word[-1] + word[:-1]) if len(word) > 1 else word

def _unrotate(word: str) -> str:
    return (word[1:] + word[0]) if len(word) > 1 else word

def encrypt_prompt(prompt: str) -> str:
    return " ".join(_rotate(w.translate(_ENC_TABLE)) for w in prompt.split())

def decrypt_prompt(cipher: str) -> str:
    return " ".join(_unrotate(w).translate(_DEC_TABLE) for w in cipher.split())

# Sanity check
_EXAMPLES = ["a cat on a mat", "hello world", "a beautiful sunset over the ocean"]
print("── Cipher sanity check ──────────────────────────────────")
for p in _EXAMPLES:
    c = encrypt_prompt(p)
    d = decrypt_prompt(c)
    print(f"  plain : {p}\n  cipher: {c}\n  back  : {d}  {'✓' if d == p else '✗'}\n")

In [ ]:
MAX_SAMPLES   = 1_000_000  # Massively scaled up
MAX_TOKEN_LEN = 77
CACHE_DIR     = "/kaggle/working/data_cache"

print("Loading massive prompt corpus (Stable Diffusion Prompts)...")
# This dataset contains 1.8 million text-to-image prompts.
sd_source = load_dataset(
    "daspartho/stable-diffusion-prompts", split="train",
    cache_dir=CACHE_DIR,
)

def get_text(row):
    return str(row.get("Prompt", row.get("prompt", row.get("text", ""))))

print(f"Dataset loaded. Total rows available: {len(sd_source):,}")

def build_pairs(dataset, max_samples: int):
    _tok = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
    pairs, seen = [], set()
    
    for row in tqdm(dataset, total=min(len(dataset), max_samples), desc="Building pairs"):
        cap = get_text(row).strip()
        if not cap or cap in seen: continue
        seen.add(cap)
        
        cipher = encrypt_prompt(cap)
        
        # Fast length checking
        if (len(_tok(cap, truncation=False)["input_ids"]) <= MAX_TOKEN_LEN and
            len(_tok(cipher, truncation=False)["input_ids"]) <= MAX_TOKEN_LEN):
            pairs.append((cap, cipher))
            
        if len(pairs) >= max_samples:
            break
            
    return pairs

# Build the massive pair list
pairs = build_pairs(sd_source, max_samples=MAX_SAMPLES)
print(f"\nGenerated {len(pairs):,} (plain, cipher) pairs for extended training.")

PAIRS_PATH = Path("/kaggle/working/pairs_extended.jsonl")
with open(PAIRS_PATH, "w") as f:
    for p, c in pairs:
        f.write(json.dumps({"plain": p, "cipher": c}) + "\n")

In [ ]:
CLIP_MODEL_ID = "openai/clip-vit-large-patch14"
BATCH_SIZE    = 256
VAL_FRAC      = 0.02
SEED          = 42

random.seed(SEED)
random.shuffle(pairs)

n_val       = max(1, int(len(pairs) * VAL_FRAC))
val_pairs   = pairs[:n_val]
train_pairs = pairs[n_val:]
print(f"Train: {len(train_pairs):,}  |  Val: {len(val_pairs):,}")

class CipherDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=77):
        self.pairs     = pairs
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        plain, cipher = self.pairs[idx]
        enc_p = self.tokenizer(plain,  padding="max_length", truncation=True,
                               max_length=self.max_len, return_tensors="pt")
        enc_c = self.tokenizer(cipher, padding="max_length", truncation=True,
                               max_length=self.max_len, return_tensors="pt")
        return {
            "plain_input_ids":      enc_p["input_ids"].squeeze(0),
            "plain_attention_mask": enc_p["attention_mask"].squeeze(0),
            "cipher_input_ids":     enc_c["input_ids"].squeeze(0),
            "cipher_attention_mask":enc_c["attention_mask"].squeeze(0),
        }

tokenizer    = CLIPTokenizer.from_pretrained(CLIP_MODEL_ID)
train_ds     = CipherDataset(train_pairs, tokenizer)
val_ds       = CipherDataset(val_pairs,   tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
from peft import PeftModel

clip_full = CLIPModel.from_pretrained(CLIP_MODEL_ID)

# ── Reference encoder: frozen, produces TARGET embeddings ────────────────────
ref_encoder = copy.deepcopy(clip_full.text_model)
ref_proj    = copy.deepcopy(clip_full.text_projection)
for p in ref_encoder.parameters(): p.requires_grad_(False)
for p in ref_proj.parameters():    p.requires_grad_(False)
ref_encoder.eval() 
ref_encoder = ref_encoder.to(DEVICE)
ref_proj    = ref_proj.to(DEVICE)

# ── RESUME LORA ENCODER ───────────────────────────────────────────────────
CKPT_PATH = "/kaggle/working/checkpoints/best_lora_epoch5_cos0.8372"
print(f"Resuming training from checkpoint: {CKPT_PATH}")

# Load the previously trained weights and explicitly set them to be trainable
lora_encoder = PeftModel.from_pretrained(
    clip_full.text_model, 
    CKPT_PATH,
    is_trainable=True  # <── THE MAGIC FLAG: Unlocks the loaded weights for training
)

lora_proj = copy.deepcopy(clip_full.text_projection)
# Keep the projection frozen so the latent space doesn't drift
for p in lora_proj.parameters(): p.requires_grad_(False)

lora_encoder.print_trainable_parameters()

n_trainable_enc = sum(p.numel() for p in lora_encoder.parameters() if p.requires_grad)
assert n_trainable_enc > 0, "FATAL: lora_encoder has no trainable parameters!"

# ── DataParallel across both T4s ──────────────────────────────────────────
if N_GPU > 1:
    lora_encoder = nn.DataParallel(lora_encoder)

lora_encoder = lora_encoder.to(DEVICE)
lora_proj    = lora_proj.to(DEVICE)

# ── Free the full model — vision encoder isn't used, don't keep it in RAM ────
import gc
del clip_full
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def cosine_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def infonce_loss(pred: torch.Tensor, target: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    p = F.normalize(pred,   dim=-1)
    t = F.normalize(target, dim=-1)
    logits  = (p @ t.T) / temperature
    targets = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, targets)

def combined_loss(pred, target, alpha=0.7):
    return alpha * cosine_loss(pred, target) + (1 - alpha) * infonce_loss(pred, target)

In [ ]:
def _unwrap(module):
    return module.module if isinstance(module, nn.DataParallel) else module

def embed_plain(input_ids, attention_mask):
    with torch.no_grad():
        out = ref_encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = _unwrap(ref_proj)(out.pooler_output)
    return F.normalize(emb, dim=-1)

def embed_cipher(input_ids, attention_mask):
    out = lora_encoder(input_ids=input_ids, attention_mask=attention_mask)
    emb = _unwrap(lora_proj)(out.pooler_output)
    return F.normalize(emb, dim=-1)

In [ ]:
LR        = 5e-5   # DECREASED: We are fine-tuning an already smart model now
EPOCHS    = 3      # 3 Epochs on 1 Million rows will take a while!
WARMUP    = 1000   # Longer warmup for the massive dataset
LOG_EVERY = 500
CKPT_DIR  = Path("/kaggle/working/checkpoints_phase2")
CKPT_DIR.mkdir(exist_ok=True)

# Train only LoRA
trainable = list(filter(lambda p: p.requires_grad, lora_encoder.parameters()))

print(f"Total trainable tensors in optimizer: {len(trainable)}")
print(f"Total trainable parameters          : {sum(p.numel() for p in trainable):,}")

# Fresh optimizer for the new phase
optimizer   = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4)
total_steps = EPOCHS * len(train_loader)
scheduler   = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=total_steps,
    pct_start=WARMUP / total_steps, anneal_strategy="cos",
)
scaler = torch.amp.GradScaler('cuda')

# Start our "best" tracker at your previous high score
best_val_cos = 0.8372 
history      = {"train_loss": [], "val_cos": []}

for epoch in range(6, 6 + EPOCHS): # Start logging at Epoch 6
    lora_encoder.train()
    running_loss  = 0.0
    epoch_loss    = 0.0
    skipped_steps = 0  

    for step, batch in enumerate(train_loader, 1):
        plain_ids  = batch["plain_input_ids"].to(DEVICE)
        plain_mask = batch["plain_attention_mask"].to(DEVICE)
        ciph_ids   = batch["cipher_input_ids"].to(DEVICE)
        ciph_mask  = batch["cipher_attention_mask"].to(DEVICE)

        with torch.no_grad():
            target_emb = embed_plain(plain_ids, plain_mask)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            pred_emb = embed_cipher(ciph_ids, ciph_mask)
            loss     = combined_loss(pred_emb, target_emb)

        scale_before = scaler.get_scale()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scale_after = scaler.get_scale()

        if scale_after >= scale_before: scheduler.step()
        else: skipped_steps += 1

        loss_val      = loss.item()
        running_loss += loss_val
        epoch_loss   += loss_val

        if step % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            running_loss = 0.0
            print(f"  Epoch {epoch} | step {step:>6}/{len(train_loader)} "
                  f"| loss {avg:.4f} | lr {scheduler.get_last_lr()[0]:.2e}"
                  + (f" | skipped {skipped_steps}" if skipped_steps else ""))

    # ── Validation ────────────────────────────────────────────────────────────
    lora_encoder.eval()
    cos_sum, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            plain_ids  = batch["plain_input_ids"].to(DEVICE)
            plain_mask = batch["plain_attention_mask"].to(DEVICE)
            ciph_ids   = batch["cipher_input_ids"].to(DEVICE)
            ciph_mask  = batch["cipher_attention_mask"].to(DEVICE)
            with torch.amp.autocast('cuda'):
                cos_sum += F.cosine_similarity(
                    embed_cipher(ciph_ids, ciph_mask),
                    embed_plain(plain_ids, plain_mask), dim=-1
                ).mean().item()
            n_batches += 1

    val_cos = cos_sum / n_batches
    history["train_loss"].append(epoch_loss / len(train_loader))
    history["val_cos"].append(val_cos)
    print(f"\n── Epoch {epoch} complete ── val cosine similarity: {val_cos:.4f}\n")

    # ── Save every epoch so training can always be resumed ───────────────────
    ckpt_path = CKPT_DIR / f"lora_epoch{epoch}_cos{val_cos:.4f}"
    _unwrap(lora_encoder).save_pretrained(str(ckpt_path))
    print(f"  ✓ Saved checkpoint → {ckpt_path}")

    if val_cos > best_val_cos:
        best_val_cos = val_cos
        print(f"  ★ New best! ({val_cos:.4f})\n")

print(f"\nPhase 2 Training complete. Best val cosine similarity: {best_val_cos:.4f}")

ETA: 2.0293333333333333333333333333333 s per step

In [ ]:
lora_encoder.eval()

test_prompts = [
    "a cat sitting on a red mat",
    "futuristic city at night with neon lights",
    "intricate sensual corset lingerie highly detailed",
]

print("══ Qualitative embedding alignment ══════════════════════")
with torch.no_grad():
    for prompt in test_prompts:
        cipher = encrypt_prompt(prompt)
        p_ids  = tokenizer(prompt, return_tensors="pt", padding="max_length", max_length=77, truncation=True).to(DEVICE)
        c_ids  = tokenizer(cipher, return_tensors="pt", padding="max_length", max_length=77, truncation=True).to(DEVICE)
        cos = F.cosine_similarity(
            embed_cipher(c_ids["input_ids"],  c_ids["attention_mask"]),
            embed_plain( p_ids["input_ids"],  p_ids["attention_mask"]), dim=-1
        ).item()
        print(f"  [{cos:.4f}]  plain : {prompt}\n           cipher: {cipher}\n")

cos_vals = []
with torch.no_grad():
    for batch in val_loader:
        plain_ids  = batch["plain_input_ids"].to(DEVICE)
        plain_mask = batch["plain_attention_mask"].to(DEVICE)
        ciph_ids   = batch["cipher_input_ids"].to(DEVICE)
        ciph_mask  = batch["cipher_attention_mask"].to(DEVICE)
        with torch.amp.autocast('cuda'):
            cos_vals.extend(
                F.cosine_similarity(
                    embed_cipher(ciph_ids, ciph_mask),
                    embed_plain(plain_ids, plain_mask), dim=-1
                ).tolist()
            )

cos_t = torch.tensor(cos_vals)
print("══ Validation set cosine similarity ══════════════════════")
print(f"  mean  : {cos_t.mean():.4f}")
print(f"  std   : {cos_t.std():.4f}")
print(f"  min   : {cos_t.min():.4f}")
print(f"  > 0.90: {(cos_t > 0.90).float().mean()*100:.1f} %")
print(f"  > 0.95: {(cos_t > 0.95).float().mean()*100:.1f} %")

In [ ]:
# Analytics
def letter_freq(text: str) -> dict:
    text  = text.lower()
    total = sum(1 for c in text if c.isalpha())
    if not total: return {}
    return {c: text.count(c) / total for c in ALPHABET}

cipher_corpus = " ".join(c for _, c in pairs[:10_000])
plain_corpus  = " ".join(p for p, _ in pairs[:10_000])

cf_cipher = letter_freq(cipher_corpus)
cf_plain  = letter_freq(plain_corpus)

uniform   = 1 / 26
kl_cipher = sum(v * math.log(v / uniform) for v in cf_cipher.values() if v > 0)
kl_plain  = sum(v * math.log(v / uniform) for v in cf_plain.values()  if v > 0)

print("══ Frequency analysis (security check) ═══════════════════")
print(f"  KL(plain  ‖ uniform) = {kl_plain:.4f}\n  KL(cipher ‖ uniform) = {kl_cipher:.4f}")
print("\n  Top-5 plain  letters: ", sorted(cf_plain.items(),  key=lambda x: -x[1])[:5])
print("  Top-5 cipher letters: ", sorted(cf_cipher.items(), key=lambda x: -x[1])[:5])

# File Saving
with open("/kaggle/working/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

with open("/kaggle/working/sample_pairs.jsonl", "w") as f:
    for p, c in pairs[:1000]:
        f.write(json.dumps({"plain": p, "cipher": c}) + "\n")

print("\nOutputs written to /kaggle/working/ -> checkpoints, metadata, and JSONs.")

In [ ]:
import os
import zipfile
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints_phase2"
OUTPUT_ZIP = WORKING_DIR / "checkpoints.zip"

def zip_folder(folder: Path, output: Path):
    if not folder.exists():
        # Auto-detect: find any folder that looks like checkpoints
        candidates = [
            d for d in WORKING_DIR.iterdir()
            if d.is_dir() and any(
                kw in d.name.lower()
                for kw in ("checkpoint", "ckpt", "model", "output", "lora")
            )
        ]
        if not candidates:
            print(f"[ERROR] No checkpoint folder found in {WORKING_DIR}")
            print("Contents:", [d.name for d in WORKING_DIR.iterdir()])
            return
        folder = candidates[0]
        print(f"[INFO] Auto-detected checkpoint folder: {folder}")

    print(f"[INFO] Zipping: {folder}  →  {output}")
    file_count = 0

    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in folder.rglob("*"):
            if file.is_file():
                arcname = file.relative_to(folder.parent)
                zf.write(file, arcname)
                file_count += 1
                print(f"  + {arcname}")

    size_mb = output.stat().st_size / 1024 / 1024
    print(f"\n[DONE] {file_count} files → {output.name} ({size_mb:.1f} MB)")

zip_folder(CHECKPOINT_DIR, OUTPUT_ZIP)